### This is a finance crew for daily stock briefs.

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the .env file from local
import openai
from dotenv import load_dotenv
env_path = r'.env'
load_dotenv(env_path)

True

In [4]:
from crewai import Agent, Task, Crew

In [ ]:
#Define Collector Agent
collector_agent = Agent(
    role="Market Data Collector",
    goal="Gather latest price movements and top headlines for {ticker}",
    backstory=(
        "You collect real-time market data and news headlines for stocks."
    ),
    allow_delegation=False,
    verbose=True
)

In [6]:
#Define Summarizer Agent
summarizer_agent = Agent(
    role="News Summarizer",
    goal="Condense news headlines into 5 key bullet points.",
    backstory=(
        "You extract the most important information from news and present it concisely."
    ),
    allow_delegation=False,
    verbose=True
)

In [7]:
#Define Risk Analyzer Agent
risk_checker_agent = Agent(
    role="Risk Analyst",
    goal="Identify key risks and unknowns from the news.",
    backstory=(
        "You flag potential risks like earnings, guidance changes, macro events, and litigation."
    ),
    allow_delegation=False,
    verbose=True
)

In [8]:
#Define Brief Writer Agent
daily_brief_agent = Agent(
    role="Daily Brief Writer",
    goal="Create a concise daily brief for {ticker}",
    backstory=(
        "You write clear, actionable daily briefs covering what changed, why, and what to watch."
    ),
    allow_delegation=False,
    verbose=True
)

In [10]:
import yfinance as yf

In [36]:
from crewai.tools import tool

In [37]:
@tool("get_stock_data")
def get_stock_data(ticker: str) -> str:
    """Get current price and recent news for a stock ticker."""
    stock = yf.Ticker(ticker)

    info = stock.info

    current_price = info.get('currentPrice', 'N/A')
    previous_close = info.get('previousClose', 'N/A')

    if current_price != 'N/A' and previous_close != 'N/A':
        change_pct = ((current_price - previous_close) / previous_close) * 100
        price_info = f'Price: ${current_price:.2f} ({change_pct:+.2f}%)'
    else:
        price_info = 'Price data unavailable'

    news = stock.news

    headlines = []

    for item in news[:8]:
        title = item['content'].get('title', '').strip()

        if title:
            headlines.append(f'- {title}')

    if not headlines:
        headlines = ['- No recent news headlines available']

    news_text = '\n'.join(headlines)

    return f'{price_info}\n\nRecent Headlines:\n{news_text}'


In [ ]:
collect_task = Task(
    description=(
        "Use the Get Stock Data tool to collect price and news for {ticker}."
        "Call the tool once and return all the data you receive."
    ),
    expected_output=(
        "Price data and news headlines returned from the tool."        
    ),
    tools=[get_stock_data],
    agent=collector_agent,
)

In [39]:
summarize_task = Task(
    description=(
        "Review the collected data and create exactly 5 bullet points."
        "If news is limited, focus on price movement and general market context."        
    ),
    expected_output=(
        "Exactly 5 bullet points summarizing key information."        
    ),    
    agent=summarizer_agent,
)

In [40]:
risk_task = Task(
    description=(
        "Based on the summary, identify 3-5 potential risks or unknowns."
        "Consider: earnings, guidance, macro trends, competition, regulation."        
    ),
    expected_output=(
        "A list of 3-5 key risks or unknowns."        
    ),    
    agent=risk_checker_agent,
)

In [41]:
brief_task = Task(
    description=(
        "Write a 6-8 line daily brief for {ticker}."
        "Structure: What changed today / Why it matters / What to watch."        
    ),
    expected_output=(
        "A 6-8 line daily brief formatted as markdown without '```'"        
    ),    
    agent=daily_brief_agent,
)

In [42]:
crewFinance = Crew(
  agents=[collector_agent, summarizer_agent, risk_checker_agent, daily_brief_agent],
  tasks=[collect_task, summarize_task, risk_task, brief_task],
  verbose=True
)

In [44]:
inputs = {
    "ticker": "GIB"    
}
result = crewFinance.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b68edfd2-720f-41d1-b06e-ec7115b11668                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the Get Stock Data tool to collect price and news for GIB.                                           │
│  Call the tool once and return all the data you receive.                                                        │
│  ID: 65358060-f22f-4739-96a7-de6c66d363b2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Data Collector                                                                                   │
│                                                                                                                 │
│  Task: Use the Get Stock Data tool to collect price and news for GIB.                                           │
│  Call the tool once and return all the data you receive.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_stock_data                                                                                           │
│  Args: {'ticker': 'GIB'}                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool get_stock_data executed with result: Price: $65.16 (-4.48%)

Recent Headlines:
- CGI Inc. (GIB) Sets Sights on Cloud Computing Opportunities in the Nordics
- CGI Expands European Sovereign Cloud And AI As Shares Struggle
- Down 9.2% in 4...

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Daily Brief Writer                                                                                      │
│                                                                                                                 │
│  Task: Write a 6-8 line daily brief for GIB.Structure: What changed today / Why it matters / What to watch.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Daily Brief Writer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **CGI Inc. (GIB) Daily Brief**                                                                                 │
│                                                                                                                 │
│  What changed today: CGI’s shares dropped sharply to $65.16, down 4.48%, marking continued weakness with a      │
│  9.2% decline over the past month. This follows RBC’s recent downgrade to Sector Perform post-Q2 2026           │
│  earnings, which triggered an 11% sell-off. Despite this, CGI is actively expanding its cloud computing and AI  │
│  services in the Nordics and Europe, signaling strategic growth initiatives.                                    │
│                                                                                                                 │
│  Why it matters: The share price volatility reflects investor concerns about earnings risks and the challenges  │
│  CGI faces in executing its cloud and AI expansion amid competitive and regulatory pressures. However, Wall     │
│  Street analysts remain bullish, projecting a potential 34% upside driven by acquisitions and AI investments.   │
│                                                                                                                 │
│  What to watch: Monitor upcoming earnings releases and guidance updates for signs of financial stability or     │
│  further risk. Also, track progress in CGI’s Nordic cloud expansion and European sovereign cloud projects, as   │
│  well as regulatory developments in the region that could impact operational costs and compliance.              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a 6-8 line daily brief for GIB.Structure: What changed today / Why it matters / What to watch.     │
│  Agent: Daily Brief Writer                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b68edfd2-720f-41d1-b06e-ec7115b11668                                                                       │
│  Final Output: **CGI Inc. (GIB) Daily Brief**                                                                   │
│                                                                                                                 │
│  What changed today: CGI’s shares dropped sharply to $65.16, down 4.48%, marking continued weakness with a      │
│  9.2% decline over the past month. This follows RBC’s recent downgrade to Sector Perform post-Q2 2026           │
│  earnings, which triggered an 11% sell-off. Despite this, CGI is actively expanding its cloud computing and AI  │
│  services in the Nordics and Europe, signaling strategic growth initiatives.                                    │
│                                                                                                                 │
│  Why it matters: The share price volatility reflects investor concerns about earnings risks and the challenges  │
│  CGI faces in executing its cloud and AI expansion amid competitive and regulatory pressures. However, Wall     │
│  Street analysts remain bullish, projecting a potential 34% upside driven by acquisitions and AI investments.   │
│                                                                                                                 │
│  What to watch: Monitor upcoming earnings releases and guidance updates for signs of financial stability or     │
│  further risk. Also, track progress in CGI’s Nordic cloud expansion and European sovereign cloud projects, as   │
│  well as regulatory developments in the region that could impact operational costs and compliance.              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯